# 🌿 Lab 2.2: Transfer Learning with ResNet50
**Module 3: Computer Vision and Image Processing**
B-Tech AI Specialization | Chitkara University | February 2026

---

## 🌾 Industry Scenario
> You have **500 images** of 5 types of plant diseases. A farmer app needs a classifier to identify diseases from phone photos. Training from scratch would take days and thousands of images. **Transfer learning** lets you adapt a model that already understands images to your specific task — quickly.

## 🎯 Objective
Fine-tune a pre-trained ResNet50 on a small plant disease dataset. Compare against training from scratch. Target: **≥80% validation accuracy in 10 epochs**.

**Time:** 120 minutes | **Mode:** Individual

---
### 📋 Lab Flow
| Stage | What happens |
|---|---|
| 🤔 Predict | Answer before coding — commit to a guess |
| 💻 Code | Fill in the `TODO` sections |
| 💡 Reveal | Click to check hint or full solution |
| 🎚️ Explore | Interactive plots — dig into your results |
---

## ⚙️ Setup — Run First

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import History

import ipywidgets as widgets
from IPython.display import display, HTML, Code
import os, time, warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow : {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("✅ Ready")

In [ ]:
def reveal_button(hint_text, solution_code):
    import ipywidgets as widgets
    from IPython.display import display, HTML, Code
    out = widgets.Output()
    hint_btn = widgets.Button(description='💡 Hint', button_style='info',
        layout=widgets.Layout(width='120px', margin='4px'))
    sol_btn  = widgets.Button(description='✅ Solution', button_style='warning',
        layout=widgets.Layout(width='140px', margin='4px'))
    hide_btn = widgets.Button(description='🙈 Hide', button_style='',
        layout=widgets.Layout(width='100px', margin='4px'))
    def on_hint(b):
        with out:
            out.clear_output(wait=True)
            display(HTML(f'<div style="background:#e3f2fd;padding:12px;border-radius:6px;'
                f'border-left:4px solid #1976D2;font-size:14px"><b>💡 Hint:</b><br>{hint_text}</div>'))
    def on_sol(b):
        with out:
            out.clear_output(wait=True)
            display(HTML('<b>✅ Solution:</b>'))
            display(Code(solution_code, language='python'))
    def on_hide(b):
        with out: out.clear_output()
    hint_btn.on_click(on_hint); sol_btn.on_click(on_sol); hide_btn.on_click(on_hide)
    display(widgets.HBox([hint_btn, sol_btn, hide_btn]), out)

print("reveal_button() ready ✅")

---
## Task 1: Prepare the Dataset

We'll use a small subset of the **PlantVillage** dataset — 5 plant disease classes, 100 images each.

### 🤔 Predict First
Before running any code, answer these:
1. We have 500 images total. How many will be in train vs. validation (80/20 split)?
2. Why do we need validation data at all — why not just train on everything?
3. What problems could arise with only 100 images per class?

In [ ]:
# ✏️ Your predictions (as comments):
# 1. Train: _400__   Validation: _100__
# 2. Validation is needed because it tells how well the model works on the unseen data.
# 3. With only 100 images per class, the risk is overfitting.

### 💻 Download and Organise the Dataset
We'll download a pre-prepared subset from a public source and set up directory structure.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import urllib.request, zipfile, os, shutil # Added shutil for dataset splitting
import tensorflow as tf # Ensure tf is imported here

# Original broken download attempt commented out
# DATA_URL  = "https://github.com/btphan95/greenr-dataset/raw/master/data.zip"
# DATA_DIR  = "plant_disease_data"
# if not os.path.exists(DATA_DIR):
#     print("Downloading dataset...")
#     urllib.request.urlretrieve(DATA_URL, "data.zip")
#     with zipfile.ZipFile("data.zip", 'r') as z:
#         z.extractall(DATA_DIR)
#     print("Extracted ✅")


# --- ALTERNATIVE: Use tf.keras.utils.get_file for flower_photos dataset as requested ---
print("Downloading flower_photos dataset using tf.keras.utils.get_file...")
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
# Download and extract the flower_photos dataset
# cache_dir ensures it's placed in /content/datasets for easier access
base_dataset_path = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True, cache_dir='/content/datasets')
print(f"Flower photos dataset downloaded to: {base_dataset_path}")

# DATA_DIR now points to the root of the raw downloaded flower dataset
DATA_DIR = base_dataset_path

# TRAIN_DIR and VAL_DIR will be set after the manual splitting in the next cells.
TRAIN_DIR = None
VAL_DIR = None

# Quick check — print class names and image counts from the raw download
if DATA_DIR and os.path.exists(DATA_DIR):
    # Filter out potential non-directory files if any
    classes = sorted([c for c in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, c))])
    print(f"\nRaw Flower dataset classes found ({len(classes)}): {classes}")
    for cls in classes:
        n = len(os.listdir(os.path.join(DATA_DIR, cls)))
        print(f"  {cls}: {n} images")

print("\nProceeding to split the dataset into train and validation sets...")

In [ ]:
reveal_button(
    hint_text="Use <code>urllib.request.urlretrieve(url, filename)</code> to download, "
              "then <code>zipfile.ZipFile</code> to extract. Set TRAIN_DIR and VAL_DIR "
              "to point at the <code>train/</code> and <code>val/</code> subdirectories.",
    solution_code=(
        "if not os.path.exists(DATA_DIR):\n"
        "    urllib.request.urlretrieve(DATA_URL, 'data.zip')\n"
        "    with zipfile.ZipFile('data.zip', 'r') as z:\n"
        "        z.extractall(DATA_DIR)\n\n"
        "TRAIN_DIR = os.path.join(DATA_DIR, 'train')\n"
        "VAL_DIR   = os.path.join(DATA_DIR, 'val')"
    )
)

---
## Task 2: Data Augmentation

With only ~80 training images per class, we need to artificially expand the dataset using **augmentation** — creating modified versions of each image on the fly during training.

### 🤔 Predict First
Look at the augmentation parameters below. For each one, predict:
- What does it do visually to the image?
- Does it make sense for plant disease photos? (Would a real phone photo look like this?)

| Parameter | Your prediction | Makes sense? |
|---|---|---|
| `horizontal_flip=True` | ? | ? |
| `rotation_range=20` | ? | ? |
| `zoom_range=0.2` | ? | ? |
| `width_shift_range=0.1` | ? | ? |

In [ ]:
# ✏️ Fill in your predictions in the table above (edit the markdown cell)
# Parameter             | Prediction                                                                | Makes sense?
# horizontal_flip       | flips the image left to right(mirror image)                               | yes
# rotation_range=20     | rotates the image by a random angle between -20 degree to 20 degree       | yes
# zoom_range=0.2        | zooms the image in or out by 20%                                          | yes
# width_shift_range=0.1 | shifts the image horizontally                                             | yes

### 💻 Build the Data Generators

In [ ]:
IMG_SIZE  = (224, 224)
BATCH_SIZE = 32

# TODO: Create an ImageDataGenerator for TRAINING with augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
)

# TODO: Create a separate generator for VALIDATION — no augmentation, just preprocessing
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# TODO: Create the flow_from_directory generators
# train_generator = None
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# val_generator = None
val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

In [ ]:
reveal_button(
    hint_text="Validation generator should have <b>no augmentation</b> — only <code>preprocessing_function</code>. "
              "Augmenting validation data would give you unrealistic accuracy scores.",
    solution_code=(
        "train_datagen = ImageDataGenerator(\n"
        "    preprocessing_function=preprocess_input,\n"
        "    horizontal_flip=True,\n"
        "    rotation_range=20,\n"
        "    zoom_range=0.2,\n"
        "    width_shift_range=0.1,\n"
        "    height_shift_range=0.1\n"
        ")\n\n"
        "val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)\n\n"
        "train_generator = train_datagen.flow_from_directory(\n"
        "    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'\n"
        ")\n"
        "val_generator = val_datagen.flow_from_directory(\n"
        "    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='categorical'\n"
        ")"
    )
)

### 🎚️ Explore: What Does Augmentation Actually Do?
Run the cell below to visualise 8 augmented versions of the same image side-by-side.

In [ ]:
# Visualise augmentation — see what the model actually trains on
sample_batch, _ = next(train_generator)
sample_img_raw  = sample_batch[0]

# Un-preprocess for display (ResNet50 uses mean subtraction, not [0,1] scaling)
def unpreprocess(img):
    img = img.copy()
    img[..., 0] += 103.939
    img[..., 1] += 116.779
    img[..., 2] += 123.68
    return np.clip(img[..., ::-1] / 255.0, 0, 1)  # BGR → RGB

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("8 Augmented Versions of the Same Image\n"
             "(What the model sees during training)", fontsize=13, fontweight='bold')

aug_gen = train_datagen.flow(
    np.expand_dims(sample_batch[0], 0), batch_size=1
)
for ax in axes.flat:
    aug_img = next(aug_gen)[0]
    ax.imshow(unpreprocess(aug_img))
    ax.axis('off')

plt.tight_layout()
plt.show()
print("✏️ Observation: How different do these look from each other?")
print("   Would you expect a plant photo from a phone to look like these?")

---
## Task 3: Build the Transfer Learning Model (Feature Extraction Phase)

Transfer learning has two phases:

```
Phase 1 — Feature Extraction:
  ResNet50 (frozen, pretrained) → GlobalAveragePooling → Dense → Softmax
  ↑ weights locked, won't change              ↑ only these train

Phase 2 — Fine-tuning:
  ResNet50 (last 20 layers UNfrozen) → GlobalAveragePooling → Dense → Softmax
  ↑ these now also update, but slowly
```

### 🤔 Predict First
1. Why do we **freeze** ResNet50's layers in Phase 1?
2. Why do we need `include_top=False`?
3. What does `GlobalAveragePooling2D` do differently from `Flatten`?

In [ ]:
# ✏️ Your predictions:
# 1. We freeze layers to keep pretrained ImageNet features and avoid overfitting while training only the new classifier.
# 2. include_top=False means removing the original 1000-class ImageNet classifier so we can add our own classifier for tomato disease classes.
# 3. GlobalAveragePooling vs Flatten:Flatten converts all pixels into a large vector causing many parameters and overfitting, while GlobalAveragePooling2D averages each feature map into one value, reducing parameters and improving generalization.

### 💻 Build the Model

In [ ]:
NUM_CLASSES = 5   # adjust if your dataset has a different number

# TODO: Load ResNet50 base — no top, pretrained on ImageNet
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# TODO: Freeze all base model layers so they don't update during Phase 1
base_model.trainable = False

# TODO: Build the full model by adding a classification head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax'),
])

# TODO: Compile with adam and categorical_crossentropy
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Check: how many layers are trainable?
trainable   = sum(1 for l in model.layers[0].layers if l.trainable)
untrainable = sum(1 for l in model.layers[0].layers if not l.trainable)
print(f"ResNet50 layers — Trainable: {trainable} | Frozen: {untrainable}")
model.summary()

In [ ]:
reveal_button(
    hint_text="Set <code>base_model.trainable = False</code> after loading. "
              "Then stack: <code>GlobalAveragePooling2D → Dense(256, relu) → Dropout(0.5) → Dense(NUM_CLASSES, softmax)</code>.",
    solution_code=(
        "base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))\n"
        "base_model.trainable = False\n\n"
        "model = models.Sequential([\n"
        "    base_model,\n"
        "    layers.GlobalAveragePooling2D(),\n"
        "    layers.Dense(256, activation='relu'),\n"
        "    layers.Dropout(0.5),\n"
        "    layers.Dense(NUM_CLASSES, activation='softmax'),\n"
        "])\n\n"
        "model.compile(\n"
        "    optimizer=optimizers.Adam(learning_rate=1e-3),\n"
        "    loss='categorical_crossentropy',\n"
        "    metrics=['accuracy']\n"
        ")"
    )
)

---
## Task 4: Phase 1 — Train the Classification Head (10 epochs)

In [ ]:
# TODO: Train the model for 10 epochs
history_phase1 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)



In [ ]:
reveal_button(
    hint_text="Call <code>model.fit(train_generator, epochs=10, validation_data=val_generator)</code>. "
              "Store the result in <code>history_phase1</code>.",
    solution_code=(
        "history_phase1 = model.fit(\n"
        "    train_generator,\n"
        "    epochs=10,\n"
        "    validation_data=val_generator\n"
        ")"
    )
)

---
## Task 5: Phase 2 — Fine-Tuning (Unfreeze Last 20 Layers)

Now we'll carefully unfreeze the **last 20 layers** of ResNet50 and train them at a very low learning rate. This lets the network adapt its deep features slightly to plant disease patterns.

### 🤔 Predict First
1. Why must the learning rate be **much lower** in fine-tuning (1e-5 vs 1e-3)?
2. Why do we unfreeze only the **last** layers, not the first?

In [ ]:
# ✏️ Your predictions:
# 1. Lower LR because pretrained ResNet50 weights should only change slightly; a high learning rate would overwrite useful ImageNet features.
# 2. Last layers because early layers learn general features (edges, colors) useful for all images, while the last layers learn task-specific plant disease patterns.

In [ ]:
# TODO: Unfreeze the last 20 layers of the base model
base_model = model.layers[0]  # get the ResNet50 sub-model

# Step 1: make base model trainable overall
base_model.trainable = True

# Step 2: freeze everything EXCEPT the last 20 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

from tensorflow.keras import optimizers



# TODO: Re-compile with a much lower learning rate
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Check how many are now trainable
trainable = sum(1 for l in base_model.layers if l.trainable)
print(f"Now trainable layers in ResNet50: {trainable}")

# TODO: Train for 5 more epochs
# history_phase2 = model.fit(...)

history_phase2 = model.fit(
    train_generator, epochs=5, validation_data=val_generator
)

In [ ]:
reveal_button(
    hint_text="<code>base_model.trainable = True</code> first, then loop: "
              "<code>for layer in base_model.layers[:-20]: layer.trainable = False</code>. "
              "Re-compile with <code>learning_rate=1e-5</code>.",
    solution_code=(
        "base_model = model.layers[0]\n"
        "base_model.trainable = True\n"
        "for layer in base_model.layers[:-20]:\n"
        "    layer.trainable = False\n\n"
        "model.compile(\n"
        "    optimizer=optimizers.Adam(learning_rate=1e-5),\n"
        "    loss='categorical_crossentropy',\n"
        "    metrics=['accuracy']\n"
        ")\n\n"
        "history_phase2 = model.fit(\n"
        "    train_generator, epochs=5, validation_data=val_generator\n"
        ")"
    )
)

---
## 🎚️ Task 6: Explore — Interactive Training Curves

Use the controls below to examine your training history. Look for:
- Where does Phase 1 plateau? Where does Phase 2 give an extra push?
- Is there a gap between train and val accuracy? What does that mean?
- At what epoch does the model first exceed 80% validation accuracy?

In [ ]:
# Build combined history from both phases
def build_history_dict(h1, h2):
    """Merge two History objects into one dict for plotting."""
    combined = {}
    for key in h1.history:
        p2_vals = h2.history.get(key, [])
        combined[key] = h1.history[key] + p2_vals
    combined['phase_boundary'] = len(h1.history['accuracy'])
    return combined

# ── Interactive curve explorer ────────────────────────────────────────────────
metric_toggle = widgets.ToggleButtons(
    options=[('Accuracy', 'accuracy'), ('Loss', 'loss')],
    description='Metric:', button_style='info'
)
show_phases = widgets.Checkbox(value=True, description='Show phase boundary')
smooth_check = widgets.Checkbox(value=False, description='Smooth curves')
out_plot = widgets.Output()

def update_curves(change=None):
    if history_phase1 is None or history_phase2 is None:
        with out_plot:
            out_plot.clear_output()
            print("⚠️  Run Tasks 4 and 5 first to generate training history.")
        return

    hist = build_history_dict(history_phase1, history_phase2)
    metric   = metric_toggle.value
    val_key  = f'val_{metric}'
    boundary = hist['phase_boundary']
    epochs   = list(range(1, len(hist[metric]) + 1))

    def smooth(vals, w=3):
        return [np.mean(vals[max(0,i-w):i+1]) for i in range(len(vals))]

    train_vals = smooth(hist[metric])     if smooth_check.value else hist[metric]
    val_vals   = smooth(hist[val_key])    if smooth_check.value else hist[val_key]

    with out_plot:
        out_plot.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))

        ax.plot(epochs, train_vals, 'b-o', markersize=5, label=f'Train {metric}', linewidth=2)
        ax.plot(epochs, val_vals,   'r-o', markersize=5, label=f'Val {metric}',   linewidth=2)

        if show_phases.value:
            ax.axvline(x=boundary + 0.5, color='purple', linestyle='--', alpha=0.7, linewidth=2)
            ymin, ymax = ax.get_ylim()
            ax.text(boundary * 0.5, ymax * 0.97, 'Phase 1\n(frozen)', ha='center',
                    color='purple', fontsize=10, fontweight='bold')
            ax.text(boundary + (len(epochs) - boundary) * 0.5, ymax * 0.97, 'Phase 2\n(fine-tune)',
                    ha='center', color='purple', fontsize=10, fontweight='bold')

        if metric == 'accuracy':
            ax.axhline(y=0.80, color='green', linestyle=':', alpha=0.8, linewidth=1.5,
                       label='80% target')
            best_val  = max(val_vals)
            best_ep   = val_vals.index(best_val) + 1
            ax.annotate(f'Best: {best_val:.1%} @ ep{best_ep}',
                        xy=(best_ep, best_val), xytext=(best_ep + 1, best_val - 0.05),
                        arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=10)

        ax.set_xlabel('Epoch', fontsize=12)
        ax.set_ylabel(metric.capitalize(), fontsize=12)
        ax.set_title(f'Training Curves — Transfer Learning with ResNet50', fontsize=13, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

metric_toggle.observe(update_curves, names='value')
show_phases.observe(update_curves, names='value')
smooth_check.observe(update_curves, names='value')

display(widgets.VBox([
    widgets.HBox([metric_toggle, show_phases, smooth_check]),
    out_plot
]))
update_curves()

---
## ✍️ Reflection

Write a short paragraph (3–5 sentences) explaining:
- Why did transfer learning work so well with only 500 images?
- What did Phase 1 learn vs Phase 2?
- Would you expect the same result if you trained from scratch? Why?

In [ ]:
# ✏️ Your reflection:
# Transfer learning worked well because the pretrained ResNet50 already learned useful visual features (edges, textures, shapes) from ImageNet, which helped our small tomato dataset achieve good accuracy.

# In Phase 1, the model learned how to use those pretrained features to classify the 5 tomato disease classes by training only the new classification head.

# In Phase 2, fine-tuning added slight adjustments to the deeper layers so the network could better recognize plant-specific disease patterns and textures.

# Training from scratch would have required a much larger dataset and likely resulted in severe overfitting and poor validation accuracy with only 500 images.

# Task
Add code cells to manually split the `flower_photos` dataset, currently located at `/tmp/.keras/datasets/flower_photos`, into `train` and `val` directories with 80/20 ratio for each class, creating the structure `/content/flower_data_split/train/<class_name>/` and `/content/flower_data_split/val/<class_name>/`, then update the `TRAIN_DIR` and `VAL_DIR` variables to point to these newly created directories respectively.

## Add cells for manual dataset splitting

### Subtask:
Manually split the flower_photos dataset into training and validation sets with an 80/20 ratio for each class.


**Reasoning**:
The current `DATA_DIR` points to `/tmp/.keras/datasets/flower_photos`, which contains a single subdirectory also named `flower_photos`. The actual image classes (like 'daisy', 'dandelion') are nested one level deeper. I need to define the correct `source_data_dir` for splitting and then implement the logic to create train and validation directories and copy files according to an 80/20 split for each class. This will involve using `os` for path manipulation, `shutil` for copying files, and `random` for shuffling image lists.



In [ ]:
import os
import shutil
import random

# Define the base directory for the split data
SPLIT_BASE_DIR = '/content/flower_data_split'

# The actual root directory containing the class folders
# Based on previous output, DATA_DIR is '/tmp/.keras/datasets/flower_photos'
# and it contains a subdirectory 'flower_photos' which has the actual classes.
original_data_root = os.path.join(DATA_DIR, 'flower_photos')

# Clean up previous split if it exists
if os.path.exists(SPLIT_BASE_DIR):
    print(f"Removing existing split directory: {SPLIT_BASE_DIR}")
    shutil.rmtree(SPLIT_BASE_DIR)

# Create the main SPLIT_BASE_DIR
os.makedirs(SPLIT_BASE_DIR, exist_ok=True)

# Create train and validation subdirectories
TRAIN_DIR = os.path.join(SPLIT_BASE_DIR, 'train')
VAL_DIR   = os.path.join(SPLIT_BASE_DIR, 'val')

os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

print(f"Created split directories: {TRAIN_DIR} and {VAL_DIR}")

# Get the list of class names (subdirectories in the original data root)
class_names = [d for d in os.listdir(original_data_root) if os.path.isdir(os.path.join(original_data_root, d))]
class_names.sort() # Ensure consistent order

print(f"Found classes: {class_names}")

# Iterate through each class and split the images
for class_name in class_names:
    class_path = os.path.join(original_data_root, class_name)
    images = [os.path.join(class_path, img) for img in os.listdir(class_path) if img.endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    # Calculate split point (80% train, 20% validation)
    split_point = int(0.8 * len(images))
    train_images = images[:split_point]
    val_images   = images[split_point:]

    # Create class subdirectories in train and val
    os.makedirs(os.path.join(TRAIN_DIR, class_name), exist_ok=True)
    os.makedirs(os.path.join(VAL_DIR, class_name), exist_ok=True)

    # Copy images to respective directories
    for img_path in train_images:
        shutil.copy(img_path, os.path.join(TRAIN_DIR, class_name, os.path.basename(img_path)))
    for img_path in val_images:
        shutil.copy(img_path, os.path.join(VAL_DIR, class_name, os.path.basename(img_path)))

    print(f"Class '{class_name}': Train {len(train_images)} images, Val {len(val_images)} images")

print("\nDataset splitting complete. Updated TRAIN_DIR and VAL_DIR.\n")

# Verify the split counts (optional)
print("Verification of split counts:")
for class_name in class_names:
    train_count = len(os.listdir(os.path.join(TRAIN_DIR, class_name)))
    val_count   = len(os.listdir(os.path.join(VAL_DIR, class_name)))
    print(f"  {class_name} - Train: {train_count}, Val: {val_count}")

# The flower_photos dataset downloaded contains categories at 'flower_photos/flower_photos/category' structure.
# The previous cell only showed one directory 'flower_photos'. Let's re-evaluate `original_data_root`
# and `class_names` to ensure we are actually getting the individual flower categories.
# If `original_data_root` itself contains the classes, the code should already handle it.
# If it is '/tmp/.keras/datasets/flower_photos/flower_photos' and then that contains 'daisy', 'dandelion' etc. then it is correct.

# Let's inspect the directory structure once more to be absolutely sure.
# The standard flower_photos dataset structure is `flower_photos/{class_name}/image.jpg`.
# The earlier output: `DATA_DIR` is `/tmp/.keras/datasets/flower_photos` and `classes` is `['flower_photos']`
# This means get_file untarred `flower_photos.tgz` into `flower_photos/flower_photos/`
# So the actual classes (daisy, dandelion, etc.) are under `DATA_DIR/flower_photos/`
# So original_data_root should correctly be `os.path.join(DATA_DIR, 'flower_photos')`.
# Let's ensure `class_names` is populated correctly. It should be if `os.listdir` works as expected.
# The example data used in the original execution only showed 1 class: 'flower_photos' with 6 images.
# This indicates that the download from `dataset_url` was either not the full dataset, or the way `get_file` extracts it is unusual.
# The current code assumes classes are direct subdirectories of `original_data_root`.
# If the download was indeed limited, the loop will run for `flower_photos` as the only class.
# I will add a check to make sure the number of classes is as expected for the actual flower_photos dataset.

# Re-checking based on common flower_photos dataset structure (5 classes)
expected_classes = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
if set(class_names) != set(expected_classes):
    print("\nWARNING: The detected classes do not match the expected flower_photos classes.")
    print("Detected classes:", class_names)
    print("Expected classes:", expected_classes)
    print("This might indicate an issue with the dataset download/extraction or a change in the dataset.")


In [ ]:
import os
!pip install nbstripout
!nbstripout *.ipynb
!git add .
!git commit -m "clean notebook metadata"
!git push

## Run subsequent cells

### Subtask:
Continue executing the remaining cells in the notebook, as the dataset preparation is now complete.


## Summary:

### Q&A
The `flower_photos` dataset was successfully split into `train` and `val` directories. The split ratio was approximately 80/20 for training and validation sets for each class. The specified directory structure `/content/flower_data_split/train/<class_name>/` and `/content/flower_data_split/val/<class_name>/` was created. The `TRAIN_DIR` and `VAL_DIR` variables were updated to point to these newly created directories respectively.

### Data Analysis Key Findings
*   The dataset was split into training and validation sets, creating new directories at `/content/flower_data_split/train` and `/content/flower_data_split/val`.
*   Five distinct flower classes were identified and processed: `daisy`, `dandelion`, `roses`, `sunflowers`, and `tulips`.
*   The images were distributed between training and validation sets for each class with an 80/20 ratio:
    *   **daisy**: 506 training images, 127 validation images.
    *   **dandelion**: 718 training images, 180 validation images.
    *   **roses**: 512 training images, 129 validation images.
    *   **sunflowers**: 559 training images, 140 validation images.
    *   **tulips**: 639 training images, 160 validation images.

### Insights or Next Steps
*   The dataset is now correctly structured and prepared for subsequent machine learning tasks, such as model training and evaluation, using the newly defined `TRAIN_DIR` and `VAL_DIR`.
*   The manual splitting process ensures a controlled and reproducible separation of data for each class, which is crucial for robust model development.
